# Session 29: DeepSeek R1 Case Study with GRPO

## 🎯 DeepSeek R1과 추론 모델

이 노트북에서는 DeepSeek R1의 학습 과정을 분석하고, **GRPO(Group Relative Policy Optimization)**를 사용하여  
Qwen2.5-1.5B 모델의 수학 추론 능력을 향상시키는 실습을 진행합니다.

### 학습 목표
- 🎯 DeepSeek R1의 학습 파이프라인 이해
- 1️⃣ GRPO 알고리즘의 상세 원리 파악
- 2️⃣ 보상 함수 설계 방법 학습
- 3️⃣ GRPO로 수학 추론 능력 향상 실습
- 4️⃣ Chain-of-Thought 추론 패턴 분석

### GPU 요구사항
- ✅ GPU 필수 (RTX 4060 8GB 이상)
- 모델: Qwen2.5-1.5B-Instruct (4bit 양자화)
- 예상 VRAM: ~5-7GB

---

In [ ]:
# 필요한 라이브러리 임포트
import torch
import gc
import json
import re
import os
import random
import numpy as np
from pathlib import Path
from datetime import datetime

# GPU 메모리 모니터링 함수
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")

print("✅ 기본 라이브러리 임포트 완료")
print(f"📅 실행 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"🔥 CUDA: {torch.version.cuda}")

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print_gpu_memory("초기 상태")
else:
    print("❌ GPU를 사용할 수 없습니다. 이 노트북은 GPU가 필수입니다.")

---
## 1️⃣ DeepSeek R1의 학습 과정 분석

### DeepSeek R1이란?

DeepSeek R1은 2024년 말 DeepSeek에서 발표한 **추론 특화 LLM**으로, OpenAI의 o1/o3에 필적하는 성능을 보여주었습니다.

### 핵심 혁신

```
🔑 핵심: 강화학습(GRPO)만으로 추론 능력을 획득!

기존 접근: SFT로 CoT 데이터 학습 → 추론 패턴 모방
DeepSeek R1: RL(GRPO)로 직접 추론 능력 발현 → 자발적 CoT 출현
```

### DeepSeek R1 학습 4단계

```
┌──────────────────────────────────────────────┐
│          DeepSeek R1 학습 파이프라인            │
├──────────────────────────────────────────────┤
│                                              │
│  Stage 1: Cold Start (소량 SFT)               │
│  └→ 기본적인 CoT 형식 학습                      │
│                                              │
│  Stage 2: 추론 중심 RL (GRPO)                  │
│  └→ 수학/코딩 문제로 추론 능력 강화              │
│  └→ 보상: 정답 여부 + 형식 보상                 │
│                                              │
│  Stage 3: Rejection Sampling + SFT            │
│  └→ RL 모델의 좋은 출력을 수집하여 재학습         │
│  └→ 일반 능력(글쓰기, 번역 등) 데이터 혼합       │
│                                              │
│  Stage 4: 전체 RL (GRPO)                      │
│  └→ 추론 + 일반 태스크 모두 보상 부여            │
│  └→ 안전성/유용성 보상 추가                     │
│                                              │
│  최종: 추론 + 일반 능력 모두 갖춘 모델            │
└──────────────────────────────────────────────┘
```

In [ ]:
# DeepSeek R1 학습 단계 상세 분석
print("=" * 60)
print("📌 DeepSeek R1 학습 파이프라인 분석")
print("=" * 60)

stages = [
    {
        "name": "Stage 1: Cold Start SFT",
        "description": "소량의 CoT 데이터로 기본 형식 학습",
        "details": [
            "수천 개의 <think>...</think> 형식 예시 학습",
            "기본적인 사고 과정 표현 방법 습득",
            "목적: RL이 탐색할 수 있는 기본 능력 확보",
        ]
    },
    {
        "name": "Stage 2: 추론 중심 RL (GRPO)",
        "description": "수학/코딩 문제로 추론 능력 강화",
        "details": [
            "GRPO 알고리즘 사용 (Value Model 불필요)",
            "보상 1: 정답 여부 (정확한 답 = +1, 오답 = 0)",
            "보상 2: 형식 보상 (<think> 태그 사용 여부)",
            "놀라운 발견: 'aha moment' - 모델이 자발적으로 자기 검증 학습!",
        ]
    },
    {
        "name": "Stage 3: Rejection Sampling + SFT",
        "description": "좋은 출력을 수집하여 재학습",
        "details": [
            "Stage 2 모델에서 정답을 맞힌 출력 수집",
            "일반 태스크(글쓰기, 번역, QA) 데이터 혼합",
            "SFT로 추론+일반 능력 통합",
        ]
    },
    {
        "name": "Stage 4: 전체 RL (GRPO)",
        "description": "모든 태스크에 대해 RL 적용",
        "details": [
            "추론 태스크: 정답 기반 보상",
            "일반 태스크: LLM-as-judge 보상",
            "안전성/유용성 보상 추가",
        ]
    },
]

for stage in stages:
    print(f"\n🔷 {stage['name']}")
    print(f"   📋 {stage['description']}")
    for detail in stage['details']:
        print(f"      • {detail}")

In [ ]:
# DeepSeek R1의 "Aha Moment" 시연
print("=" * 60)
print("📌 DeepSeek R1의 'Aha Moment' 예시")
print("=" * 60)

print("""
🧠 "Aha Moment"란?
   RL 학습 중 모델이 자발적으로 자기 검증(self-verification) 능력을 획득하는 순간

📝 예시 - 수학 문제 풀이 과정:

<think>
문제: 27 × 14 = ?

풀이 시작:
27 × 14 = 27 × (10 + 4)
        = 27 × 10 + 27 × 4  
        = 270 + 108
        = 378

잠깐, 검증해보자.            ← 🎯 Aha Moment!
27 × 14 = (30 - 3) × 14    ← 다른 방법으로 검증
        = 420 - 42
        = 378 ✓              ← 일치 확인
</think>

27 × 14 = 378

💡 핵심: 이 자기 검증 능력은 SFT로 학습시킨 것이 아니라,
   RL(GRPO)를 통해 자연스럽게 발현된 것입니다!
   
   → 정답을 맞히면 보상을 받으므로, 검증 과정이 정답률을 높이는 데
     도움이 된다는 것을 모델이 스스로 학습한 것입니다.
""")

---
## 2️⃣ GRPO 알고리즘 상세 (Group Relative Policy Optimization)

### GRPO 수학적 정의

```
GRPO 목적 함수:

J(θ) = E_q~P(Q) [ 1/G × Σ_{i=1}^{G} (
    min(
        r_i(θ) × Â_i,
        clip(r_i(θ), 1-ε, 1+ε) × Â_i
    ) - β × KL(π_θ || π_ref)
)]

여기서:
- q: 입력 프롬프트
- G: 그룹 크기 (각 프롬프트당 생성 개수)
- r_i(θ) = π_θ(o_i|q) / π_old(o_i|q): 정책 비율
- Â_i = (r_i - mean(r)) / std(r): 정규화된 이점
- ε: 클리핑 범위
- β: KL 페널티 강도
```

### PPO와의 핵심 차이

```
PPO:  Advantage = Value Function으로 계산 → Value Model 필요
GRPO: Advantage = 그룹 내 상대 비교로 계산 → Value Model 불필요!
```

### GRPO 알고리즘 의사코드

```python
for each batch of prompts Q:
    for each prompt q in Q:
        # 1. G개 응답 생성
        outputs = [generate(q) for _ in range(G)]
        
        # 2. 보상 계산
        rewards = [reward_fn(q, o) for o in outputs]
        
        # 3. 그룹 내 정규화
        advantages = (rewards - mean(rewards)) / std(rewards)
        
    # 4. PPO 스타일 업데이트 (클리핑 + KL 페널티)
    update_policy(outputs, advantages)
```

In [ ]:
# GRPO 알고리즘 시뮬레이션
print("=" * 60)
print("📌 GRPO 알고리즘 시뮬레이션")
print("=" * 60)

def simulate_grpo_step(prompt, group_size=4):
    """
    GRPO 한 스텝을 시뮬레이션합니다.
    """
    print(f"\n📝 프롬프트: {prompt['question']}")
    print(f"   정답: {prompt['answer']}")
    print(f"   그룹 크기: {group_size}")
    
    # 1. G개 응답 생성 (시뮬레이션)
    print(f"\n   Step 1: {group_size}개 응답 생성")
    outputs = []
    for i in range(group_size):
        # 일부는 정답, 일부는 오답으로 시뮬레이션
        correct = random.random() > 0.4  # 60% 정답률
        if correct:
            answer = prompt['answer']
        else:
            answer = str(int(prompt['answer']) + random.choice([-2, -1, 1, 2]))
        outputs.append({"answer": answer, "correct": str(answer) == str(prompt['answer'])})
        status = "✅" if outputs[-1]["correct"] else "❌"
        print(f"      응답 {i+1}: {answer} {status}")
    
    # 2. 보상 계산
    print(f"\n   Step 2: 보상 계산")
    rewards = []
    for o in outputs:
        r = 1.0 if o["correct"] else 0.0
        rewards.append(r)
        print(f"      r = {r:.1f}")
    
    rewards = np.array(rewards)
    
    # 3. 그룹 내 정규화
    print(f"\n   Step 3: 그룹 내 정규화")
    mean_r = np.mean(rewards)
    std_r = np.std(rewards) + 1e-8
    advantages = (rewards - mean_r) / std_r
    
    print(f"      평균: {mean_r:.3f}, 표준편차: {std_r:.3f}")
    for i, (adv, r) in enumerate(zip(advantages, rewards)):
        direction = "↑ 강화" if adv > 0 else "↓ 약화" if adv < 0 else "→ 유지"
        print(f"      Â_{i+1} = ({r:.1f} - {mean_r:.3f}) / {std_r:.3f} = {adv:+.3f} ({direction})")
    
    # 4. 정책 업데이트 방향
    print(f"\n   Step 4: 정책 업데이트 방향")
    for i, (o, adv) in enumerate(zip(outputs, advantages)):
        if adv > 0:
            print(f"      응답 {i+1} ({o['answer']}): 확률 증가 ↑ (정답)")
        elif adv < 0:
            print(f"      응답 {i+1} ({o['answer']}): 확률 감소 ↓ (오답)")
    
    return advantages

# 시뮬레이션 실행
random.seed(42)
prompts = [
    {"question": "15 + 27 = ?", "answer": "42"},
    {"question": "8 × 7 = ?", "answer": "56"},
]

for p in prompts:
    print("\n" + "=" * 50)
    simulate_grpo_step(p, group_size=4)

---
## 3️⃣ 환경 설정 및 Qwen2.5-1.5B 로드

In [ ]:
# 추가 라이브러리 임포트
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

print("✅ 모든 라이브러리 임포트 완료")
print("   • transformers, peft, trl (GRPOTrainer), datasets")

In [ ]:
# 모델 설정
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = Path("./outputs/grpo_training")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 4bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("=" * 60)
print("📌 GRPO 학습 설정")
print("=" * 60)
print(f"   모델: {MODEL_NAME}")
print(f"   양자화: 4bit NF4")
print(f"   출력: {OUTPUT_DIR}")

In [ ]:
# 모델 및 토크나이저 로드
print("=" * 60)
print("📌 모델 로딩")
print("=" * 60)

print("🔄 토크나이저 로딩...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
print(f"   ✅ 토크나이저 로드 완료")

print("🔄 모델 로딩...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print(f"   ✅ 모델 로드 완료")
print_gpu_memory("모델 로드 후")

In [ ]:
# LoRA 설정
print("=" * 60)
print("📌 LoRA 설정")
print("=" * 60)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

print(f"   LoRA rank: {lora_config.r}")
print(f"   LoRA alpha: {lora_config.lora_alpha}")
print(f"   Target: {lora_config.target_modules}")

---
## 4️⃣ 수학 문제 데이터 준비 (간단한 산술/논리 문제)

GRPO 학습에 사용할 수학 문제 데이터를 준비합니다.  
검증 가능한 정답이 있는 문제를 사용합니다.

In [ ]:
# 수학 문제 데이터 생성
print("=" * 60)
print("📌 수학 문제 데이터 생성")
print("=" * 60)

random.seed(42)

def generate_math_problems(n=100):
    """
    다양한 유형의 수학 문제를 생성합니다.
    """
    problems = []
    
    for _ in range(n):
        problem_type = random.choice(["addition", "multiplication", "mixed", "word_problem"])
        
        if problem_type == "addition":
            a = random.randint(10, 99)
            b = random.randint(10, 99)
            question = f"{a} + {b}의 값은 얼마인가요?"
            answer = str(a + b)
            
        elif problem_type == "multiplication":
            a = random.randint(2, 15)
            b = random.randint(2, 15)
            question = f"{a} × {b}의 값은 얼마인가요?"
            answer = str(a * b)
            
        elif problem_type == "mixed":
            a = random.randint(2, 20)
            b = random.randint(2, 10)
            c = random.randint(1, 20)
            question = f"{a} × {b} + {c}의 값은 얼마인가요?"
            answer = str(a * b + c)
            
        else:  # word_problem
            templates = [
                {
                    "template": "사과가 {a}개 있었는데 {b}개를 더 샀습니다. 총 사과는 몇 개인가요?",
                    "calc": lambda a, b: a + b
                },
                {
                    "template": "한 상자에 연필이 {a}자루씩 들어있습니다. {b}상자에는 총 몇 자루의 연필이 있나요?",
                    "calc": lambda a, b: a * b
                },
                {
                    "template": "쿠키가 {total}개 있습니다. {n}명이 똑같이 나누면 한 사람당 몇 개인가요?",
                    "calc": lambda total, n: total // n
                },
            ]
            t = random.choice(templates[:2])  # 나눗셈은 정수 결과만
            a = random.randint(3, 20)
            b = random.randint(2, 10)
            question = t["template"].format(a=a, b=b)
            answer = str(t["calc"](a, b))
        
        # GRPO용 프롬프트 형식
        prompt = (
            f"다음 문제를 단계별로 풀어주세요. "
            f"먼저 풀이 과정을 보여주고, 마지막에 '정답: [숫자]' 형식으로 답을 제시하세요.\n\n"
            f"문제: {question}"
        )
        
        problems.append({
            "prompt": prompt,
            "answer": answer,
            "type": problem_type,
        })
    
    return problems

# 문제 생성
math_problems = generate_math_problems(100)

print(f"\n✅ {len(math_problems)}개 수학 문제 생성 완료")
print(f"\n📊 유형별 분포:")
type_counts = {}
for p in math_problems:
    type_counts[p['type']] = type_counts.get(p['type'], 0) + 1
for t, c in type_counts.items():
    print(f"   {t}: {c}개")

print(f"\n📋 예시 문제들:")
for p in math_problems[:5]:
    print(f"   [{p['type']}] {p['prompt'][:60]}... → 정답: {p['answer']}")

In [ ]:
# GRPO 데이터셋 준비
print("=" * 60)
print("📌 GRPO 데이터셋 준비")
print("=" * 60)

# GRPOTrainer가 기대하는 형식으로 변환
# prompt 필드만 필요 (보상은 보상 함수로 계산)
grpo_data = []
for p in math_problems:
    messages = [
        {"role": "system", "content": "당신은 수학 문제를 단계별로 풀어주는 AI입니다. 풀이 과정을 보여주고, 마지막에 '정답: [숫자]' 형식으로 답을 제시하세요."},
        {"role": "user", "content": p["prompt"]}
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    grpo_data.append({
        "prompt": prompt_text,
        "answer": p["answer"],  # 보상 함수에서 사용
    })

grpo_dataset = Dataset.from_list(grpo_data)

print(f"✅ GRPO 데이터셋 생성 완료: {len(grpo_dataset)}개 샘플")
print(f"📋 컬럼: {grpo_dataset.column_names}")
print(f"\n📋 첫 번째 프롬프트 미리보기:")
print(grpo_dataset[0]["prompt"][:200] + "...")

---
## 5️⃣ 보상 함수 정의 (정답 검증)

GRPO의 핵심은 **보상 함수(Reward Function)**입니다.  
DeepSeek R1처럼 규칙 기반 보상을 정의합니다.

### 보상 설계

| 보상 | 조건 | 점수 |
|------|------|------|
| 정답 보상 | 최종 답이 정답과 일치 | +1.0 |
| 형식 보상 | '정답: [숫자]' 형식 준수 | +0.5 |
| 풀이 보상 | 풀이 과정이 존재 | +0.25 |
| 오답 패널티 | 오답 | 0.0 |

In [ ]:
# 보상 함수 정의
print("=" * 60)
print("📌 보상 함수 정의")
print("=" * 60)

def extract_answer(text: str) -> str:
    """
    모델 출력에서 정답을 추출합니다.
    '정답: 42' 또는 '정답: [42]' 형식에서 숫자를 추출
    """
    # 패턴 1: '정답: 숫자' 형식
    patterns = [
        r'정답\s*[:：]\s*\[?(\-?\d+)\]?',
        r'답\s*[:：]\s*\[?(\-?\d+)\]?',
        r'결과\s*[:：]\s*\[?(\-?\d+)\]?',
        r'=\s*(\-?\d+)\s*$',  # 마지막 줄의 = 숫자
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, text)
        if matches:
            return matches[-1]  # 마지막 매치 사용
    
    # 패턴 매칭 실패 시 마지막 숫자 추출
    numbers = re.findall(r'\b(\d+)\b', text)
    if numbers:
        return numbers[-1]
    
    return ""


def reward_function(completions, answer, **kwargs):
    """
    GRPO 보상 함수
    
    Args:
        completions: 모델이 생성한 응답 리스트
        answer: 정답 리스트
    
    Returns:
        rewards: 보상 점수 리스트
    """
    rewards = []
    
    for completion, correct_answer in zip(completions, answer):
        # 텍스트 추출
        if isinstance(completion, list):
            text = completion[-1]["content"] if completion else ""
        else:
            text = str(completion)
        
        reward = 0.0
        
        # 1. 정답 보상 (+1.0)
        extracted = extract_answer(text)
        if extracted == str(correct_answer):
            reward += 1.0
        
        # 2. 형식 보상 (+0.5): '정답:' 패턴 사용
        if re.search(r'정답\s*[:：]', text):
            reward += 0.5
        
        # 3. 풀이 보상 (+0.25): 충분한 풀이 과정
        if len(text) > 50:  # 최소 50자 이상의 풀이
            reward += 0.25
        
        rewards.append(reward)
    
    return rewards


print("✅ 보상 함수 정의 완료")
print("\n📋 보상 체계:")
print("   • 정답: +1.0")
print("   • 형식 준수 ('정답: [숫자]'): +0.5")
print("   • 풀이 과정 존재 (50자+): +0.25")
print("   • 최대 보상: 1.75")

# 보상 함수 테스트
print("\n📊 보상 함수 테스트:")
test_completions = [
    "15 + 27을 계산해보겠습니다.\n15 + 27 = 42\n\n정답: 42",
    "42",
    "음... 잘 모르겠습니다.",
    "15 + 27 = 43이므로 정답: 43",
]
test_answers = ["42", "42", "42", "42"]

test_rewards = reward_function(test_completions, test_answers)
for comp, rew in zip(test_completions, test_rewards):
    print(f"   '{comp[:50]}...' → 보상: {rew:.2f}")

---
## 6️⃣ GRPO 학습 실행 (GRPOTrainer from trl)

TRL의 GRPOTrainer를 사용하여 GRPO 학습을 실행합니다.

In [ ]:
# 학습 전 모델 성능 측정 (베이스라인)
print("=" * 60)
print("📌 학습 전 베이스라인 성능 측정")
print("=" * 60)

def evaluate_math_model(model, tokenizer, problems, n_eval=10):
    """수학 문제에 대한 모델 성능 평가"""
    model.eval()
    correct = 0
    total = min(n_eval, len(problems))
    results = []
    
    for i in range(total):
        prompt = problems[i]["prompt"]
        correct_answer = problems[i]["answer"]
        
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        extracted = extract_answer(response)
        is_correct = extracted == str(correct_answer)
        
        if is_correct:
            correct += 1
        
        results.append({
            "correct_answer": correct_answer,
            "extracted": extracted,
            "is_correct": is_correct,
            "response": response[:200],
        })
    
    accuracy = correct / total * 100
    return accuracy, results

# 베이스라인 평가
print("🔄 베이스라인 성능 측정 중...")
baseline_acc, baseline_results = evaluate_math_model(model, tokenizer, grpo_data, n_eval=10)

print(f"\n📊 베이스라인 정확도: {baseline_acc:.1f}%")
print(f"\n📋 상세 결과:")
for i, r in enumerate(baseline_results, 1):
    status = "✅" if r["is_correct"] else "❌"
    print(f"   {i}. 정답: {r['correct_answer']}, 추출: {r['extracted']}, {status}")
    print(f"      응답: {r['response'][:100]}...")

print_gpu_memory("베이스라인 평가 후")

In [ ]:
# GRPO 학습 설정 (RTX 4060 최적화)
print("=" * 60)
print("📌 GRPO 학습 설정")
print("=" * 60)

grpo_config = GRPOConfig(
    output_dir=str(OUTPUT_DIR / "grpo_checkpoint"),
    
    # 학습 설정
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    
    # GRPO 특화 설정
    num_generations=4,           # 그룹 크기 (G) - 메모리 제한으로 4
    max_completion_length=256,   # 최대 생성 길이
    max_prompt_length=512,       # 최대 프롬프트 길이
    
    # 메모리 최적화
    fp16=True,
    optim="paged_adamw_8bit",
    
    # 로깅
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",
    
    # 기타
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
)

print("📋 GRPO 핵심 설정:")
print(f"   num_generations (G): {grpo_config.num_generations}")
print(f"   max_completion_length: {grpo_config.max_completion_length}")
print(f"   batch_size: {grpo_config.per_device_train_batch_size}")
print(f"   gradient_accumulation: {grpo_config.gradient_accumulation_steps}")
print(f"   learning_rate: {grpo_config.learning_rate}")
print(f"   epochs: {grpo_config.num_train_epochs}")

print("\n💡 RTX 4060 메모리 제한으로:")
print("   • num_generations=4 (DeepSeek은 64 사용)")
print("   • max_completion_length=256")
print("   • per_device_train_batch_size=1")

In [ ]:
# GRPO Trainer 생성 및 학습
print("=" * 60)
print("📌 GRPO 학습 시작")
print("=" * 60)
print_gpu_memory("학습 시작 전")

# GRPOTrainer 생성
grpo_trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=grpo_dataset,
    reward_funcs=reward_function,
    tokenizer=tokenizer,
    peft_config=lora_config,
)

print("\n🔥 GRPO 학습 시작...")
print("   (이 과정은 RTX 4060에서 60-90분 소요될 수 있습니다)")

grpo_result = grpo_trainer.train()

print(f"\n✅ GRPO 학습 완료!")
print(f"   Training Loss: {grpo_result.training_loss:.4f}")
print(f"   Steps: {grpo_result.global_step}")
print_gpu_memory("학습 완료 후")

In [ ]:
# GRPO 모델 저장
print("🔄 GRPO 모델 저장 중...")
grpo_trainer.save_model(str(OUTPUT_DIR / "grpo_checkpoint"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "grpo_checkpoint"))
print(f"✅ GRPO 모델 저장 완료: {OUTPUT_DIR / 'grpo_checkpoint'}")

---
## 7️⃣ 학습 전후 추론 능력 비교

In [ ]:
# 학습 후 성능 측정
print("=" * 60)
print("📌 학습 전후 추론 능력 비교")
print("=" * 60)

# 학습 후 평가
print("🔄 학습 후 성능 측정 중...")
grpo_trainer.model.eval()

# 새로운 테스트 문제 (학습에 사용하지 않은)
test_problems_new = []
random.seed(999)  # 다른 시드로 새 문제 생성
for _ in range(10):
    a = random.randint(10, 50)
    b = random.randint(10, 50)
    op = random.choice(["+", "×"])
    if op == "+":
        answer = str(a + b)
    else:
        answer = str(a * b)
    
    messages = [
        {"role": "system", "content": "당신은 수학 문제를 단계별로 풀어주는 AI입니다. 풀이 과정을 보여주고, 마지막에 '정답: [숫자]' 형식으로 답을 제시하세요."},
        {"role": "user", "content": f"다음 문제를 단계별로 풀어주세요. 먼저 풀이 과정을 보여주고, 마지막에 '정답: [숫자]' 형식으로 답을 제시하세요.\n\n문제: {a} {op} {b}의 값은 얼마인가요?"}
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    test_problems_new.append({"prompt": prompt_text, "answer": answer})

# 평가 실행
after_acc, after_results = evaluate_math_model(
    grpo_trainer.model, tokenizer, test_problems_new, n_eval=10
)

print(f"\n📊 성능 비교:")
print(f"   학습 전 (베이스라인): {baseline_acc:.1f}%")
print(f"   학습 후 (GRPO):      {after_acc:.1f}%")
improvement = after_acc - baseline_acc
print(f"   향상:                {improvement:+.1f}%")

print(f"\n📋 학습 후 상세 결과:")
for i, r in enumerate(after_results, 1):
    status = "✅" if r["is_correct"] else "❌"
    print(f"   {i}. 정답: {r['correct_answer']}, 추출: {r['extracted']}, {status}")
    print(f"      응답: {r['response'][:150]}...")

---
## 8️⃣ Chain-of-Thought 추론 패턴 분석

GRPO 학습 후 모델이 어떤 추론 패턴을 보이는지 분석합니다.

In [ ]:
# Chain-of-Thought 패턴 분석
print("=" * 60)
print("📌 Chain-of-Thought 추론 패턴 분석")
print("=" * 60)

# 상세 분석용 문제
analysis_problems = [
    "37 + 48의 값은 얼마인가요?",
    "12 × 9의 값은 얼마인가요?",
    "사과가 15개 있었는데 8개를 더 샀습니다. 총 사과는 몇 개인가요?",
]

for prompt_text in analysis_problems:
    print(f"\n{'='*50}")
    print(f"❓ 문제: {prompt_text}")
    print(f"{'='*50}")
    
    messages = [
        {"role": "system", "content": "당신은 수학 문제를 단계별로 풀어주는 AI입니다. 풀이 과정을 보여주고, 마지막에 '정답: [숫자]' 형식으로 답을 제시하세요."},
        {"role": "user", "content": f"다음 문제를 단계별로 풀어주세요. 먼저 풀이 과정을 보여주고, 마지막에 '정답: [숫자]' 형식으로 답을 제시하세요.\n\n문제: {prompt_text}"}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(grpo_trainer.model.device)
    
    with torch.no_grad():
        outputs = grpo_trainer.model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.3,  # 낮은 온도로 안정적 출력
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"\n🤖 모델 응답:")
    print(f"   {response}")
    
    # 패턴 분석
    print(f"\n📊 패턴 분석:")
    has_steps = bool(re.search(r'[1-9]\.|단계|먼저|그다음', response))
    has_answer_format = bool(re.search(r'정답\s*[:：]', response))
    has_calculation = bool(re.search(r'[=×+\-÷]', response))
    response_length = len(response)
    
    print(f"   단계적 풀이: {'✅' if has_steps else '❌'}")
    print(f"   정답 형식: {'✅' if has_answer_format else '❌'}")
    print(f"   계산 과정: {'✅' if has_calculation else '❌'}")
    print(f"   응답 길이: {response_length}자")

In [ ]:
# GRPO 학습의 영향 분석
print("=" * 60)
print("📌 GRPO 학습 영향 분석")
print("=" * 60)

print("""
📊 GRPO 학습으로 기대되는 변화:

1️⃣ 정답률 향상
   • 정답에 대한 보상(+1.0)으로 정확한 계산 능력 강화
   • 오답 패턴의 확률 감소

2️⃣ 형식 준수율 향상
   • 형식 보상(+0.5)으로 '정답: [숫자]' 패턴 학습
   • 일관된 출력 형식 유지

3️⃣ 풀이 과정 생성
   • 풀이 보상(+0.25)으로 단계적 설명 능력 향상
   • Chain-of-Thought 패턴 발현 가능

4️⃣ DeepSeek R1과의 차이
   • 규모: R1은 671B, 우리는 1.5B (약 450배 차이)
   • 데이터: R1은 수십만 문제, 우리는 100문제
   • 그룹 크기: R1은 G=64, 우리는 G=4
   • 학습 시간: R1은 수천 GPU-hour, 우리는 ~1시간
   
   → 하지만 GRPO의 핵심 원리는 동일합니다!
   → 더 많은 데이터와 더 큰 모델로 확장하면 성능이 크게 향상됩니다.
""")

In [ ]:
# 학습 로그 분석
print("=" * 60)
print("📌 GRPO 학습 로그 분석")
print("=" * 60)

if hasattr(grpo_trainer, 'state') and grpo_trainer.state.log_history:
    log_history = grpo_trainer.state.log_history
    
    print("\n📊 학습 이력:")
    for entry in log_history:
        if 'loss' in entry:
            step = entry.get('step', '?')
            loss = entry.get('loss', '?')
            reward = entry.get('reward', entry.get('rewards/mean', '?'))
            print(f"   Step {step}: loss={loss}" + 
                  (f", reward={reward:.3f}" if isinstance(reward, float) else ""))
    
    print("\n📊 GRPO 특화 메트릭 설명:")
    print("   • loss: GRPO 정책 손실")
    print("   • reward: 그룹 평균 보상 (높을수록 정답률 향상)")
    print("   • kl: 참조 모델 대비 KL 발산 (너무 높으면 과적합)")
else:
    print("\n⚠️ 학습 로그를 찾을 수 없습니다.")

In [ ]:
# GPU 메모리 정리
print("🧹 GPU 메모리 최종 정리...")
print_gpu_memory("정리 전")

del grpo_trainer
del model
gc.collect()
torch.cuda.empty_cache()

print_gpu_memory("정리 후")
print("✅ 메모리 정리 완료")

---
## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

1️⃣ **DeepSeek R1 학습 과정**: Cold Start SFT → 추론 RL → Rejection Sampling + SFT → 전체 RL

2️⃣ **GRPO 알고리즘**: 그룹 내 상대 비교로 Value Model 없이 학습. 메모리 효율적.

3️⃣ **보상 함수 설계**: 정답 보상(+1.0) + 형식 보상(+0.5) + 풀이 보상(+0.25)

4️⃣ **RTX 4060 GRPO 설정**:
   - `num_generations=4` (그룹 크기)
   - `per_device_train_batch_size=1`
   - `gradient_accumulation_steps=8`
   - `max_completion_length=256`

5️⃣ **Chain-of-Thought 패턴**: GRPO 학습으로 모델이 단계적 추론 패턴을 학습

6️⃣ **"Aha Moment"**: 충분한 학습 시 모델이 자발적으로 자기 검증 능력을 획득

### Part 5 강화학습 전체 요약

| 세션 | 주제 | 핵심 |
|------|------|------|
| Session 26 | RL 개념 | PPO/DPO/GRPO 원리 이해 |
| Session 27 | Preference 데이터 | 선호도 데이터 수집/생성 |
| Session 28 | DPO 학습 | SFT → DPO 파이프라인 실습 |
| Session 29 | GRPO 학습 | DeepSeek R1 사례, 수학 추론 |

### 다음 파트 예고

- 📘 **Part 6**: 배포 & 평가
  - 양자화, API 서빙, Streamlit 웹앱, 평가 메트릭

In [ ]:
# 최종 확인
print("=" * 60)
print("📌 최종 확인")
print("=" * 60)

# 저장된 체크포인트 확인
checkpoint_dir = OUTPUT_DIR / "grpo_checkpoint"
if checkpoint_dir.exists():
    files = list(checkpoint_dir.glob("*"))
    total_size = sum(f.stat().st_size for f in files if f.is_file()) / 1024**2
    print(f"  ✅ GRPO 체크포인트: {len(files)} 파일, {total_size:.1f}MB")
else:
    print(f"  ⚠️ GRPO 체크포인트: 디렉토리 없음")

print(f"\n📊 학습 결과 요약:")
print(f"   베이스라인 정확도: {baseline_acc:.1f}%")
print(f"   GRPO 후 정확도:   {after_acc:.1f}%")
print(f"   향상:             {after_acc - baseline_acc:+.1f}%")

print("\n" + "=" * 60)
print("✅ Session 29 완료!")
print("   DeepSeek R1 Case Study와 GRPO 학습을 성공적으로 실습했습니다.")
print("   Part 5 (강화학습) 전체 과정이 완료되었습니다!")